# Lockdown activity — stage 4 chart (own custom analysis)

Builds the primary + companion chart designed in `analysis-log.md` stage 4, for the
stage-1 proposition:

> Chat activity was highest during the COVID lockdown periods (2020–2021, within the
> available data) and has shown a declining trend since restrictions lifted — rather
> than staying flat or increasing over time.

Loads the already-anonymised, already-featured export from
[`01.3-your-own-chat.ipynb`](../lesson1/01.3-your-own-chat.ipynb) (same `config.toml`
`current` file, same `TimeFeatures`/`RegexFeature` columns) — no re-parsing here, this
notebook only adds the period lookup and the two stage-4 charts on top.

In [ ]:
import pandas as pd

from goad_toolkit.datatransforms import FlagDates, GroupAgg, Pipeline
from goad_toolkit.visualizer import FacetPlot, LinePlot, PlotSettings

from wa_analyzer.data import load_own_chat

own = load_own_chat()
own.shape

## Reference periods

Copied from `analysis-log.md` stage 2 — the same static, approximate lookup, not
recomputed here. Still unvalidated against an authoritative timeline (flagged in stage 2
and again in stage 4's sensitivity-check list); treat the shaded bands below as
approximate, not exact.

In [ ]:
LOCKDOWN_PERIODS = [
    ("Partial tightening pre-lockdown", "2020-10-14", "2020-12-13"),
    ("Lockdown 2020–2021", "2020-12-14", "2021-04-28"),
    ("Lockdown 2021–2022", "2021-12-19", "2022-01-14"),
]

# Stage 2's pipeline step 4: join the lookup as a categorical flag, for any later
# during-vs-outside comparison -- not used by the trend charts below, which shade the
# periods directly instead.
lockdown_dates = pd.concat(
    [pd.Series(pd.date_range(start, end)) for _, start, end in LOCKDOWN_PERIODS]
)
own = Pipeline().add(
    FlagDates, column="timestamp", dates=lockdown_dates, feature="is_lockdown_period"
).apply(own)
own["is_lockdown_period"].mean()

## Weekly bins

Stage 3/4 decision: weekly, not daily — spike days (birthdays, bad-news response
spikes) get labelled separately rather than smoothed into the line, so weekly binning
keeps the trend legible without hiding those spikes inside a rolling average.

`week_start` is a real date (the Monday of each ISO week), not the `year_week` string
`TimeFeatures` already added — needed so the lockdown shading below can line up against
actual calendar dates on the x-axis.

In [ ]:
own["week_start"] = pd.to_datetime(own["timestamp"]).dt.tz_localize(None).dt.to_period("W").dt.start_time

weekly = Pipeline().add(GroupAgg, by="week_start", agg="size", feature="messages").apply(own)
weekly.head()

## Primary chart: weekly volume, lockdown windows shaded

The single comparison this plot exists to make (stage 4): does weekly message volume
peak during lockdown and decline afterward, or is it flat/noisy/rising?

In [ ]:
settings = PlotSettings(
    figsize=(12, 5),
    title="Weekly message volume, Aug 2020 – Aug 2026",
    xlabel="",
    ylabel="messages per week",
    xtick_rotation=45,
)
lines = LinePlot(settings)
fig, ax = lines.plot(data=weekly, x="week_start", y="messages", color="#333333")

for i, (label, start, end) in enumerate(LOCKDOWN_PERIODS):
    ax.axvspan(
        pd.Timestamp(start), pd.Timestamp(end),
        color="crimson", alpha=0.15,
        label="Lockdown / restriction period" if i == 0 else None,
    )
ax.legend()

## Companion chart: same trend, split per author

Stage 4's robustness check — confirms the pattern holds across all 9 people rather than
being driven by 1–2, tying back to stage 3's "how many independent units back this"
question. Not the parked per-user curiosity itself, just a check on this claim.

In [ ]:
weekly_by_author = Pipeline().add(
    GroupAgg, by=["week_start", "author"], agg="size", feature="messages"
).apply(own)

facet_settings = PlotSettings(
    figsize=(14, 10),
    title="Weekly message volume per author",
    ylabel="messages per week",
    xtick_rotation=45,
    sharey=True,
    max_cols=3,
)
facet = FacetPlot(facet_settings)
fig, axes = facet.plot(
    inner=LinePlot(facet_settings), data=weekly_by_author, by="author",
    x="week_start", y="messages", color="#333333",
)
for ax in axes:
    for start, end in [(s, e) for _, s, e in LOCKDOWN_PERIODS]:
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), color="crimson", alpha=0.15)

## Next: stage 5 (critique)

Look at both charts above — away and back, the way `goad_analysis_checklist`'s
stage-5 table asks — before the next chat message answers its questions. That
interview needs your own read of the picture, not a description of what the code above
was trying to do.